# Kapitan 04: refs, the secrets story

A ref is a reference in the inventory, `?{backend:path||function}`, resolved by a backend:
`plain` and `base64` for demos, `gpg`, `gkms`, `awskms`, `azkms`, `vaultkv`, `vaulttransit` for
real. Functions like `random:str` create the value once; `embed-refs: true` in `.kapitan` writes
the encrypted or encoded blob into the compiled output so `kapitan refs --reveal` needs no
side channel.


In [ ]:
cd /source/work/kapitan-reference
export HOME=/tmp
find system/refs -type f | sort | head -12; echo; cat system/refs/targets/hello/api-token 2>/dev/null | head -5


In [ ]:
cd /source/work/kapitan-reference
grep -B2 -A3 'api-token' compiled/hello/manifests/hello-secret.yml | head -12


In [ ]:
cd /source/work/kapitan-reference
kapitan refs --reveal -f compiled/hello/manifests/hello-secret.yml 2>/dev/null | yq '.data'


Writing a ref by hand, for a value that must not be random: the token name encodes the backend and the path, the target gives it the recipient configuration.


In [ ]:
cd /source/work/kapitan-reference
printf 'hunter2' | kapitan refs --write plain:targets/hello/admin-password -t hello -f - && cat system/refs/targets/hello/admin-password && echo && kapitan refs --reveal --ref-file system/refs/targets/hello/admin-password; echo


Real backends look identical from the inventory's point of view. The `tesoro` target shows the in-cluster variant: the Tesoro admission controller reveals refs at apply time, so even the rendered manifests never contain a value.


In [ ]:
cd /source/work/kapitan-reference
grep -rn 'gkms\|vaultkv\|awskms' inventory/classes | head -5; echo; sed -n 1,25p inventory/classes/features/tesoro.yml


In [ ]:
cd /source/work/kapitan-reference
kapitan lint --search-secrets --skip-class-checks --skip-yamllint 2>&1 | tail -5 || true


Try it: write a second plain ref, reveal it, then delete the ref file and read what compile says.
